In [ ]:
!pip install -q -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 36.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
print("HF token loaded")

HF token loaded


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

print("Setting up 4-bit quantization config...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()
print("✅ Mistral-7B-Instruct loaded in 4-bit")

Setting up 4-bit quantization config...
Loading tokenizer for mistralai/Mistral-7B-Instruct-v0.2...
Loading model...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

✅ Mistral-7B-Instruct loaded in 4-bit


In [ ]:
from datasets import load_dataset
import random

print("Loading TriviaQA validation set...")
triviaqa_val = load_dataset("trivia_qa", "rc.nocontext", split="validation")
print(f"Full validation set: {len(triviaqa_val)} examples")

# Fixed random sample of 500 for reproducibility
random.seed(42)
eval_indices = random.sample(range(len(triviaqa_val)), 500)
eval_set = triviaqa_val.select(eval_indices)
print(f"Evaluation slice: {len(eval_set)} examples")

# Show one example
ex = eval_set[0]
print(f"\nExample question: {ex['question']}")
print(f"Gold answer: {ex['answer']['value']}")
print(f"First 3 aliases: {ex['answer']['aliases'][:3]}")

Loading TriviaQA validation set...


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/138384 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/17944 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/17210 [00:00<?, ? examples/s]

Full validation set: 17944 examples
Evaluation slice: 500 examples

Example question: What was first worn by British soldiers in India in 1845?
Gold answer: Khaki
First 3 aliases: ['Khaki', 'Kackeys', 'Kackey pants']


In [ ]:
def generate_answer(prompt: str, max_new_tokens: int = 40) -> str:
    """Generate an answer from Mistral given a prompt."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # greedy, deterministic
            pad_token_id=tokenizer.pad_token_id,
        )
    # Strip the input prompt from the output
    generated = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return generated.strip()

# Sanity test
test_prompt = "[INST] What is the capital of France? [/INST] Answer:"
print("Test generation:", generate_answer(test_prompt, max_new_tokens=20))

Test generation: The capital city of France is Paris. Paris is one of the most famous cities in the world and


In [ ]:
def build_zeroshot_prompt(question: str) -> str:
    """Zero-shot prompt using Mistral's instruction template."""
    return f"[INST] Answer the following trivia question with a short, concise answer (just the answer itself, no explanation).\n\nQuestion: {question} [/INST] Answer:"


FEW_SHOT_EXAMPLES = [
    ("Which planet is known as the Red Planet?", "Mars"),
    ("Who painted the Mona Lisa?", "Leonardo da Vinci"),
    ("What is the chemical symbol for gold?", "Au"),
    ("In what year did World War II end?", "1945"),
    ("What is the largest ocean on Earth?", "Pacific Ocean"),
]

def build_fewshot_prompt(question: str) -> str:
    """Few-shot prompt with 5 Q/A exemplars."""
    examples_text = "\n\n".join([f"Question: {q}\nAnswer: {a}" for q, a in FEW_SHOT_EXAMPLES])
    return f"[INST] Answer trivia questions with a short, concise answer. Here are some examples:\n\n{examples_text}\n\nNow answer this question:\nQuestion: {question} [/INST] Answer:"

# Test both
print("=== Zero-shot test ===")
print(generate_answer(build_zeroshot_prompt("Who wrote Hamlet?"), max_new_tokens=30))
print("\n=== Few-shot test ===")
print(generate_answer(build_fewshot_prompt("Who wrote Hamlet?"), max_new_tokens=30))

=== Zero-shot test ===
William Shakespeare

=== Few-shot test ===
William Shakespeare wrote Hamlet. It is one of his most famous works and is considered a classic of English literature. Hamlet is a tragedy about a


In [ ]:
from tqdm import tqdm
import json

print("Running Baseline 1 (zero-shot) on 500 TriviaQA questions...")

baseline1_results = []
for ex in tqdm(eval_set, desc="Baseline 1"):
    question = ex["question"]
    gold = ex["answer"]["value"]
    aliases = ex["answer"]["aliases"]

    prompt = build_zeroshot_prompt(question)
    prediction = generate_answer(prompt, max_new_tokens=40)

    baseline1_results.append({
        "question": question,
        "gold": gold,
        "aliases": aliases,
        "prediction": prediction,
    })

# Save to Drive
with open("/content/drive/MyDrive/baseline1_results.json", "w") as f:
    json.dump(baseline1_results, f, indent=2)

print(f"\n Baseline 1 done. Saved {len(baseline1_results)} predictions to Drive.")

# Show first 3 examples
for r in baseline1_results[:3]:
    print(f"\nQ: {r['question']}")
    print(f"Gold: {r['gold']}")
    print(f"Pred: {r['prediction']}")

Running Baseline 1 (zero-shot) on 500 TriviaQA questions...


Baseline 1: 100%|██████████| 500/500 [14:04<00:00,  1.69s/it]



 Baseline 1 done. Saved 500 predictions to Drive.

Q: What was first worn by British soldiers in India in 1845?
Gold: Khaki
Pred: Khaki Uniforms

Q: Which US gangster was released from Alcatraz prison in November 1939?
Gold: Al Capone
Pred: Robert Stroud, but the commonly held belief is that it was actually Robert Stroud ( alias "The Birdman of Alcatraz" ) aka Robert Stroud Jr. who escaped. However

Q: Of which tribe of Red Indians was Geronimo a chief
Gold: Apache
Pred: Geronimo was a chief of the Chiricahua Apache tribe.


In [ ]:
print("Running Baseline 2 (few-shot) on 500 TriviaQA questions...")

baseline2_results = []
for ex in tqdm(eval_set, desc="Baseline 2"):
    question = ex["question"]
    gold = ex["answer"]["value"]
    aliases = ex["answer"]["aliases"]

    prompt = build_fewshot_prompt(question)
    prediction = generate_answer(prompt, max_new_tokens=40)

    baseline2_results.append({
        "question": question,
        "gold": gold,
        "aliases": aliases,
        "prediction": prediction,
    })

# Save to Drive
with open("/content/drive/MyDrive/baseline2_results.json", "w") as f:
    json.dump(baseline2_results, f, indent=2)

print(f"\n Baseline 2 done. Saved {len(baseline2_results)} predictions to Drive.")

# Show first 3 examples
for r in baseline2_results[:3]:
    print(f"\nQ: {r['question']}")
    print(f"Gold: {r['gold']}")
    print(f"Pred: {r['prediction']}")

Running Baseline 2 (few-shot) on 500 TriviaQA questions...


Baseline 2: 100%|██████████| 500/500 [16:35<00:00,  1.99s/it]


 Baseline 2 done. Saved 500 predictions to Drive.

Q: What was first worn by British soldiers in India in 1845?
Gold: Khaki
Pred: The first uniforms made entirely of khaki material were worn by British soldiers in India in 1845. Khaki is a type of brownish-green color and was chosen for

Q: Which US gangster was released from Alcatraz prison in November 1939?
Gold: Al Capone
Pred: Robert Stroud, who is believed to be the gangster you're asking about, was not actually released from Alcatraz prison. In fact, there is no evidence that he was ever held

Q: Of which tribe of Red Indians was Geronimo a chief
Gold: Apache
Pred: Geronimo was a chief of the Bedonkohe band of the Apache tribe.


In [ ]:
import re
import string

def normalize_answer(s):
    """SQuAD-style normalization: lowercase, strip punctuation/articles/extra whitespace."""
    s = s.lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = "".join(ch for ch in s if ch not in string.punctuation)
    s = " ".join(s.split())
    return s

def quick_em_strict(pred, gold, aliases):
    """Strict: normalized prediction equals normalized gold/alias exactly."""
    pred_norm = normalize_answer(pred)
    candidates = [gold] + list(aliases)
    return int(any(normalize_answer(c) == pred_norm for c in candidates if c))

def quick_em_loose(pred, gold, aliases):
    """Loose: any normalized gold/alias appears as substring in normalized prediction."""
    pred_norm = normalize_answer(pred)
    candidates = [gold] + list(aliases)
    return int(any(normalize_answer(c) in pred_norm for c in candidates if normalize_answer(c)))

def quick_f1(pred, gold, aliases):
    """Token-level F1 against the best-matching gold/alias."""
    pred_tokens = normalize_answer(pred).split()
    if not pred_tokens:
        return 0.0
    best_f1 = 0.0
    for c in [gold] + list(aliases):
        gold_tokens = normalize_answer(c).split()
        if not gold_tokens:
            continue
        common = set(pred_tokens) & set(gold_tokens)
        if not common:
            continue
        precision = len(common) / len(pred_tokens)
        recall = len(common) / len(gold_tokens)
        f1 = 2 * precision * recall / (precision + recall)
        best_f1 = max(best_f1, f1)
    return best_f1

# Compute for both
print("="*60)
print("BASELINE RESULTS — 500 TriviaQA validation questions")
print("="*60)

for name, results in [("Baseline 1 (zero-shot)", baseline1_results),
                      ("Baseline 2 (few-shot) ", baseline2_results)]:
    em_strict = sum(quick_em_strict(r["prediction"], r["gold"], r["aliases"]) for r in results) / len(results)
    em_loose = sum(quick_em_loose(r["prediction"], r["gold"], r["aliases"]) for r in results) / len(results)
    f1 = sum(quick_f1(r["prediction"], r["gold"], r["aliases"]) for r in results) / len(results)
    print(f"\n{name}")
    print(f"  Strict EM: {em_strict:.3f}  ({em_strict*100:.1f}%)")
    print(f"  Loose EM:  {em_loose:.3f}  ({em_loose*100:.1f}%)")
    print(f"  F1 Score:  {f1:.3f}  ({f1*100:.1f}%)")

BASELINE RESULTS — 500 TriviaQA validation questions

Baseline 1 (zero-shot)
  Strict EM: 0.358  (35.8%)
  Loose EM:  0.746  (74.6%)
  F1 Score:  0.492  (49.2%)

Baseline 2 (few-shot) 
  Strict EM: 0.292  (29.2%)
  Loose EM:  0.728  (72.8%)
  F1 Score:  0.426  (42.6%)


In [ ]:
eval_code = '''
"""
eval_quick.py — Quick EM/F1 evaluation used by Mayur for interim baselines.
Adwait is writing the canonical eval; this is here for reproducibility.
"""
import re
import string
import json


def normalize_answer(s):
    s = s.lower()
    s = re.sub(r"\\b(a|an|the)\\b", " ", s)
    s = "".join(ch for ch in s if ch not in string.punctuation)
    s = " ".join(s.split())
    return s


def em_strict(pred, gold, aliases):
    pred_norm = normalize_answer(pred)
    candidates = [gold] + list(aliases)
    return int(any(normalize_answer(c) == pred_norm for c in candidates if c))


def em_loose(pred, gold, aliases):
    pred_norm = normalize_answer(pred)
    candidates = [gold] + list(aliases)
    return int(any(normalize_answer(c) in pred_norm for c in candidates if normalize_answer(c)))


def f1_score(pred, gold, aliases):
    pred_tokens = normalize_answer(pred).split()
    if not pred_tokens:
        return 0.0
    best = 0.0
    for c in [gold] + list(aliases):
        gold_tokens = normalize_answer(c).split()
        if not gold_tokens:
            continue
        common = set(pred_tokens) & set(gold_tokens)
        if not common:
            continue
        p = len(common) / len(pred_tokens)
        r = len(common) / len(gold_tokens)
        best = max(best, 2 * p * r / (p + r))
    return best


def evaluate(results):
    n = len(results)
    strict = sum(em_strict(r["prediction"], r["gold"], r["aliases"]) for r in results) / n
    loose = sum(em_loose(r["prediction"], r["gold"], r["aliases"]) for r in results) / n
    f1 = sum(f1_score(r["prediction"], r["gold"], r["aliases"]) for r in results) / n
    return {"strict_em": strict, "loose_em": loose, "f1": f1, "n": n}


if __name__ == "__main__":
    for path in ["baseline1_results.json", "baseline2_results.json"]:
        with open(path) as f:
            data = json.load(f)
        scores = evaluate(data)
        print(f"{path}: {scores}")
'''

with open('/content/drive/MyDrive/eval_quick.py', 'w') as f:
    f.write(eval_code)

print("Saved eval_quick.py to Drive")

Saved eval_quick.py to Drive
